# PlantVillage Balance Dataset

This notebook creates a capped version of `dataset/raw/PlantVillage` using
the following rule:

- if a class has `<= 1000` images, keep all of them
- if a class has `> 1000` images, randomly keep `1000`

The output is written to:

- `dataset/raw/balance dataset`


In [1]:
from __future__ import annotations

from pathlib import Path
import random
import shutil

import pandas as pd


In [2]:
BASE_DIR = Path.cwd().resolve().parent
SOURCE_DIR = BASE_DIR / 'dataset' / 'raw' / 'PlantVillage'
TARGET_DIR = BASE_DIR / 'dataset' / 'raw' / 'balance dataset'
MAX_IMAGES_PER_CLASS = 1000
RANDOM_SEED = 42

assert SOURCE_DIR.exists(), SOURCE_DIR
SOURCE_DIR, TARGET_DIR


(PosixPath('/home/moeenuddin/Desktop/Deep_learning/RAGAge/RAG_Diagnostic_Agent/dataset/raw/PlantVillage'),
 PosixPath('/home/moeenuddin/Desktop/Deep_learning/RAGAge/RAG_Diagnostic_Agent/dataset/raw/balance dataset'))

In [3]:
def list_class_files(class_dir: Path) -> list[Path]:
    """Return sorted image files inside a class directory."""
    return sorted(path for path in class_dir.iterdir() if path.is_file())


class_file_map = {
    class_dir.name: list_class_files(class_dir)
    for class_dir in sorted(SOURCE_DIR.iterdir())
    if class_dir.is_dir()
}

source_counts_df = (
    pd.Series(
        {class_name: len(files) for class_name, files in class_file_map.items()},
        name='source_count',
    )
    .rename_axis('class_name')
    .reset_index()
)
source_counts_df


,class_name,source_count
0,Pepper__bell___Bacterial_spot,997
1,Pepper__bell___healthy,1478
2,Potato___Early_blight,1000
3,Potato___Late_blight,1000
4,Potato___healthy,152
5,Tomato_Bacterial_spot,2127
6,Tomato_Early_blight,1000
7,Tomato_Late_blight,1909
8,Tomato_Leaf_Mold,952
9,Tomato_Septoria_leaf_spot,1771


In [4]:
rng = random.Random(RANDOM_SEED)

selected_files_by_class: dict[str, list[Path]] = {}
for class_name, files in class_file_map.items():
    if len(files) <= MAX_IMAGES_PER_CLASS:
        selected_files_by_class[class_name] = files
    else:
        selected_files_by_class[class_name] = sorted(
            rng.sample(files, MAX_IMAGES_PER_CLASS),
            key=lambda path: path.name,
        )

selection_summary_df = pd.DataFrame(
    {
        'class_name': list(selected_files_by_class.keys()),
        'selected_count': [
            len(files) for files in selected_files_by_class.values()
        ],
    }
)

selection_summary_df = source_counts_df.merge(selection_summary_df, on='class_name')
selection_summary_df['removed_count'] = (
    selection_summary_df['source_count'] - selection_summary_df['selected_count']
)
selection_summary_df.sort_values('class_name').reset_index(drop=True)


,class_name,source_count,selected_count,removed_count
0,Pepper__bell___Bacterial_spot,997,997,0
1,Pepper__bell___healthy,1478,1000,478
2,Potato___Early_blight,1000,1000,0
3,Potato___Late_blight,1000,1000,0
4,Potato___healthy,152,152,0
5,Tomato_Bacterial_spot,2127,1000,1127
6,Tomato_Early_blight,1000,1000,0
7,Tomato_Late_blight,1909,1000,909
8,Tomato_Leaf_Mold,952,952,0
9,Tomato_Septoria_leaf_spot,1771,1000,771


In [5]:
if TARGET_DIR.exists():
    raise FileExistsError(
        f'{TARGET_DIR} already exists. Remove it first if you want a fresh export.'
    )

for class_name, files in selected_files_by_class.items():
    class_target_dir = TARGET_DIR / class_name
    class_target_dir.mkdir(parents=True, exist_ok=True)
    for source_file in files:
        shutil.copy2(source_file, class_target_dir / source_file.name)

print(f'Balanced dataset created at: {TARGET_DIR}')


Balanced dataset created at: /home/moeenuddin/Desktop/Deep_learning/RAGAge/RAG_Diagnostic_Agent/dataset/raw/balance dataset


In [6]:
balanced_counts_df = pd.DataFrame(
    {
        'class_name': sorted(selected_files_by_class.keys()),
        'balanced_count': [
            len(selected_files_by_class[class_name])
            for class_name in sorted(selected_files_by_class.keys())
        ],
    }
)

balanced_counts_df


,class_name,balanced_count
0,Pepper__bell___Bacterial_spot,997
1,Pepper__bell___healthy,1000
2,Potato___Early_blight,1000
3,Potato___Late_blight,1000
4,Potato___healthy,152
5,Tomato_Bacterial_spot,1000
6,Tomato_Early_blight,1000
7,Tomato_Late_blight,1000
8,Tomato_Leaf_Mold,952
9,Tomato_Septoria_leaf_spot,1000


In [7]:
print(f'Total classes: {len(selected_files_by_class)}')
print(f'Total images in balanced dataset: {balanced_counts_df['balanced_count'].sum():,}')


Total classes: 15
Total images in balanced dataset: 13,474
